In [ ]:
import pickle
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# Load the batch results
with open("batch_results_composite_levy.pkl", "rb") as f:
    results = pickle.load(f)
print("Finished Loading")
# Convert to DataFrame
df = pd.DataFrame(results)


In [ ]:
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_avg_total_visits_over_clock_interval(df: pd.DataFrame, clock_start: str, clock_end: str):
    """
    Plot the average total number of bee visits per plant (by x-coordinate)
    between two real clock times, assuming each timestep = 1 second and simulation starts at 9:00:00 AM.

    Args:
        df (pd.DataFrame): DataFrame from loaded batch_results.
        clock_start (str): Start time in "H:M:S" format (e.g., "9:45:00").
        clock_end (str): End time in "H:M:S" format (e.g., "13:45:01").
    """

    sim_start_time = datetime.strptime("09:00:00", "%H:%M:%S")
    t_start = int((datetime.strptime(clock_start, "%H:%M:%S") - sim_start_time).total_seconds())
    t_end = int((datetime.strptime(clock_end, "%H:%M:%S") - sim_start_time).total_seconds())

    # For title formatting
    title_time_str = f"{clock_start.lower()} to {clock_end.lower()}"

    # Filter for plant agents within the timestep interval
    plant_df = df[
        (df["Agent Type"] == "plant") &
        (df["Step"] >= t_start) &
        (df["Step"] < t_end)
    ].copy()

    # Convert Bee Positions from string to tuple if needed
    if isinstance(plant_df["Bee Positions"].iloc[0], str):
        plant_df["Bee Positions"] = plant_df["Bee Positions"].apply(eval)

    # Extract x-coordinate
    plant_df["x"] = plant_df["Bee Positions"].apply(lambda pos: round(pos[0], 2))

    # Ensure numeric visit count
    plant_df["Plant visit count"] = pd.to_numeric(plant_df["Plant visit count"], errors="coerce")

    # For each plant (by RunId and x), get the max count during the interval
    visit_totals = (
        plant_df.groupby(["RunId", "x"])["Plant visit count"]
        .max()
        .reset_index(name="total_visits")
    )

    # Average across runs
    avg_visits = visit_totals.groupby("x")["total_visits"].mean().reset_index()

    # Plot
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=avg_visits, x="x", y="total_visits", marker="o")
    plt.title(f"Average Total Bee Visits per Plant ({title_time_str})")
    plt.xlabel("Plant X Coordinate")
    plt.ylabel("Average Total Visits")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_avg_total_visits_over_clock_interval(df, "9:00:00", "9:01:00")


In [ ]:
9:16:40 9:18:39